# Project 01 — Online Retail Sales: Cleaning & EDA

This notebook uses the **UCI Online Retail** dataset: real transactional data from a UK-based non-store retailer covering **1 Dec 2010 to 9 Dec 2011**.

**Source:** UCI Machine Learning Repository, dataset 352  
**DOI:** 10.24432/C5BW33  
**License:** CC BY 4.0

## Goals
1. Download/load the source data reproducibly.
2. Profile data quality.
3. Separate cancellations/returns from completed sales.
4. Clean fields and engineer analysis-ready features.
5. Explore revenue, orders, customers, products, countries, and seasonality.
6. Export a Power BI-ready CSV.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Works when the notebook is launched from either the repository root or notebooks/
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = RAW_DIR / "online_retail.zip"
XLSX_PATH = RAW_DIR / "Online Retail.xlsx"

UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

if not XLSX_PATH.exists():
    print("Downloading UCI Online Retail dataset...")
    urlretrieve(UCI_ZIP_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(RAW_DIR)

df_raw = pd.read_excel(XLSX_PATH)
print(f"Loaded {len(df_raw):,} rows and {df_raw.shape[1]} columns.")
df_raw.head()


## 1. Data-quality profile

We first inspect schema, missing values, duplicate rows, numeric ranges, and cancellation indicators. This is deliberately done **before** cleaning so the notebook preserves an audit trail.


In [ ]:
profile = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "unique": df_raw.nunique(dropna=True)
})
profile


In [ ]:
print("Exact duplicate rows:", f"{df_raw.duplicated().sum():,}")
print("Date range:", df_raw["InvoiceDate"].min(), "to", df_raw["InvoiceDate"].max())
print("Quantity range:", df_raw["Quantity"].min(), "to", df_raw["Quantity"].max())
print("UnitPrice range:", df_raw["UnitPrice"].min(), "to", df_raw["UnitPrice"].max())

invoice = df_raw["InvoiceNo"].astype("string")
print("Cancellation rows:", f"{invoice.str.startswith('C', na=False).sum():,}")
print("Non-positive quantity rows:", f"{(df_raw['Quantity'] <= 0).sum():,}")
print("Non-positive price rows:", f"{(df_raw['UnitPrice'] <= 0).sum():,}")


## 2. Cleaning strategy

For the **completed-sales** analytical table used by Power BI we:
- standardize column names;
- remove exact duplicates;
- flag cancellation invoices before filtering;
- retain only positive quantity and positive unit price;
- drop rows without a product description;
- keep missing customer IDs rather than deleting valid anonymous sales;
- create `Revenue = Quantity × UnitPrice`;
- derive date dimensions useful in Power BI.

Cancellations/returns are excluded from the main sales table, but we report their counts separately so they are not silently lost.


In [ ]:
df = df_raw.copy()

df.columns = [
    "invoice_no", "stock_code", "description", "quantity",
    "invoice_date", "unit_price", "customer_id", "country"
]

# Normalize types/text
df["invoice_no"] = df["invoice_no"].astype("string").str.strip()
df["stock_code"] = df["stock_code"].astype("string").str.strip()
df["description"] = df["description"].astype("string").str.strip()
df["country"] = df["country"].astype("string").str.strip()
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64").astype("string")

rows_start = len(df)
duplicates = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

df["is_cancellation"] = df["invoice_no"].str.startswith("C", na=False)
cancel_rows = int(df["is_cancellation"].sum())

sales = df.loc[
    (~df["is_cancellation"])
    & (df["quantity"] > 0)
    & (df["unit_price"] > 0)
    & df["description"].notna()
    & df["invoice_date"].notna()
].copy()

sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["date"] = sales["invoice_date"].dt.date
sales["year"] = sales["invoice_date"].dt.year
sales["month"] = sales["invoice_date"].dt.month
sales["month_name"] = sales["invoice_date"].dt.month_name().str[:3]
sales["year_month"] = sales["invoice_date"].dt.to_period("M").astype(str)
sales["day_of_week"] = sales["invoice_date"].dt.day_name()
sales["hour"] = sales["invoice_date"].dt.hour

print(f"Starting rows: {rows_start:,}")
print(f"Exact duplicates removed: {duplicates:,}")
print(f"Cancellation rows excluded: {cancel_rows:,}")
print(f"Completed-sales rows retained: {len(sales):,}")
print(f"Rows with missing customer_id retained: {sales['customer_id'].isna().sum():,}")


In [ ]:
sales.describe(include="all").T


## 3. KPI snapshot

These are useful validation numbers before building Power BI measures.


In [ ]:
kpis = pd.Series({
    "Revenue": sales["revenue"].sum(),
    "Orders": sales["invoice_no"].nunique(),
    "Known Customers": sales["customer_id"].nunique(),
    "Products": sales["stock_code"].nunique(),
    "Countries": sales["country"].nunique(),
    "Units Sold": sales["quantity"].sum(),
    "Avg Order Value": sales.groupby("invoice_no")["revenue"].sum().mean()
})
kpis


## 4. Monthly sales trend


In [ ]:
monthly = (
    sales.groupby("year_month", as_index=False)
    .agg(revenue=("revenue", "sum"),
         orders=("invoice_no", "nunique"),
         units=("quantity", "sum"))
)
monthly["avg_order_value"] = monthly["revenue"] / monthly["orders"]
monthly.head()


In [ ]:
ax = monthly.plot(x="year_month", y="revenue", figsize=(12, 5), marker="o", legend=False)
ax.set_title("Monthly Revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. Country performance

The retailer is UK-based, so the United Kingdom is expected to dominate. We show non-UK markets separately to make international patterns easier to see.


In [ ]:
country_summary = (
    sales.groupby("country", as_index=False)
    .agg(revenue=("revenue", "sum"),
         orders=("invoice_no", "nunique"),
         customers=("customer_id", "nunique"))
    .sort_values("revenue", ascending=False)
)
country_summary.head(15)


In [ ]:
non_uk = country_summary[country_summary["country"] != "United Kingdom"].head(10).sort_values("revenue")
ax = non_uk.plot.barh(x="country", y="revenue", figsize=(10, 6), legend=False)
ax.set_title("Top 10 Non-UK Markets by Revenue")
ax.set_xlabel("Revenue (£)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 6. Product performance

Rank products by revenue and quantity. Product descriptions can vary slightly for the same stock code, so Power BI modeling should use `stock_code` as the product key.


In [ ]:
product_summary = (
    sales.groupby(["stock_code", "description"], as_index=False)
    .agg(revenue=("revenue", "sum"),
         units=("quantity", "sum"),
         orders=("invoice_no", "nunique"))
    .sort_values("revenue", ascending=False)
)
product_summary.head(15)


## 7. Customer analysis

Only rows with a known customer ID are used for customer-level metrics. Anonymous transactions remain in the overall sales table and KPIs.


In [ ]:
known = sales[sales["customer_id"].notna()].copy()

customer_summary = (
    known.groupby("customer_id", as_index=False)
    .agg(revenue=("revenue", "sum"),
         orders=("invoice_no", "nunique"),
         units=("quantity", "sum"),
         first_purchase=("invoice_date", "min"),
         last_purchase=("invoice_date", "max"))
)
customer_summary["avg_order_value"] = customer_summary["revenue"] / customer_summary["orders"]
customer_summary.sort_values("revenue", ascending=False).head(15)


## 8. Order-value distribution and outlier awareness

Large wholesale transactions can create genuine extreme values. We **do not automatically delete outliers**. Instead, we inspect them and leave business-rule decisions explicit.


In [ ]:
order_summary = (
    sales.groupby("invoice_no", as_index=False)
    .agg(order_revenue=("revenue", "sum"),
         items=("quantity", "sum"),
         lines=("stock_code", "size"),
         invoice_date=("invoice_date", "min"),
         country=("country", "first"))
)

order_summary["order_revenue"].describe(percentiles=[.5, .75, .9, .95, .99])


In [ ]:
upper = order_summary["order_revenue"].quantile(0.99)
ax = order_summary.loc[order_summary["order_revenue"] <= upper, "order_revenue"].plot.hist(
    bins=50, figsize=(10, 5)
)
ax.set_title("Order Revenue Distribution (up to 99th percentile)")
ax.set_xlabel("Order Revenue (£)")
plt.tight_layout()
plt.show()


## 9. Weekday and hourly purchasing patterns


In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = (
    sales.groupby("day_of_week")["revenue"].sum()
    .reindex(weekday_order)
    .dropna()
)
ax = weekday.plot.bar(figsize=(10, 5))
ax.set_title("Revenue by Day of Week")
ax.set_xlabel("")
ax.set_ylabel("Revenue (£)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
hourly = sales.groupby("hour")["revenue"].sum()
ax = hourly.plot(figsize=(10, 5), marker="o")
ax.set_title("Revenue by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("Revenue (£)")
plt.tight_layout()
plt.show()


## 10. Export Power BI-ready data

The exported CSV is the main fact table for the first dashboard. It can be loaded directly into Power BI. For a more advanced model, create separate Date, Product, Customer, and Country dimensions in Power Query or Python.


In [ ]:
OUTPUT = PROCESSED_DIR / "online_retail_clean.csv"
sales.to_csv(OUTPUT, index=False)

print(f"Saved {len(sales):,} rows to: {OUTPUT}")
print(f"File size: {OUTPUT.stat().st_size / 1024**2:,.1f} MB")


## Suggested Power BI measures

After loading `online_retail_clean.csv`, create measures such as:

```DAX
Total Revenue = SUM(online_retail_clean[revenue])

Total Orders = DISTINCTCOUNT(online_retail_clean[invoice_no])

Units Sold = SUM(online_retail_clean[quantity])

Known Customers = DISTINCTCOUNT(online_retail_clean[customer_id])

Average Order Value = DIVIDE([Total Revenue], [Total Orders])
```

### Recommended dashboard pages
- **Executive Overview:** Revenue, Orders, Units, Customers, AOV, monthly trend
- **Product Performance:** Top products, revenue contribution, units
- **Customer Analytics:** top customers, repeat orders, customer value
- **Geography:** UK vs international markets and country performance
- **Time Analysis:** month, weekday, and hour patterns

### Next analytical extensions
- RFM customer segmentation
- cohort/retention analysis
- market-basket analysis
- cancellation/return analysis
- anomaly detection and demand forecasting
